In [2]:
import json
import time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

In [3]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    ##api_key="9d9240b6592a464399b88dafe710da9c.U00kh_iANaeR5HPen36I0rUd"
    api_key="ollama"
)

In [4]:
INCLUSION = """
    1. The study proposes, evaluates, or applies one or more Large Language Models (LLMs) (e.g., GPT, Llama, Gemma, Mistral, Qwen, Claude, PaLM, DeepSeek, or other transformer-based foundation models) as a primary component of the research.
    2. The study is conducted within an educational context.
    3. The study addresses one or more information extraction (IE) tasks from educational data, such as Named Entity Recognition (NER), Relation Extraction (RE), Attribute extraction, Knowledge extraction, and Structured data generation.
    4. The study is published in a peer-reviewed venue (e.g., journal and conference).
    5. 
    """

EXCLUSION = """
    1. The study did not propose, evaluate, or apply one or more Large Language Models (LLMs) (e.g., GPT, Llama, Gemma, Mistral, Qwen, Claude, PaLM, DeepSeek, or other transformer-based foundation models) as a primary component of the research.
    2. The study is not conducted within an educational context.
    3. The study did not address one or more information extraction (IE) tasks from educational data, such as Named Entity Recognition (NER), Relation Extraction (RE), Attribute extraction, Knowledge extraction, and Structured data generation.
"""

In [5]:
df = pd.read_csv(r"C:\Users\ikaqu\Desktop\PROJECT\automatic-SLR\dataset\Scopus.csv")

In [6]:
df.head(5)

,ID,Title,Year,Source title,Abstract,Author Keywords,Document Type
0,1,Enhancing Few-Shot Biomedical Relation Extract...,2025,2025 4th International Conference on Artificia...,Relation extraction plays a key role in biomed...,few-shot learning; large language models; prom...,Conference paper
1,2,AI-assisted interpretation of Markush structur...,2026,Journal of Cheminformatics,Automated interpretation of Markush structures...,Cheminformatics; Information extraction; Large...,Review
2,3,Knowledge graph-enhanced large language models...,2026,Reliability Engineering and System Safety,Crane accidents are characterized by high sudd...,Causal analysis; Crane accidents; Explainable ...,Article
3,4,19th International Conference on Document Anal...,2026,Lecture Notes in Computer Science,The proceedings contain 148 papers. The specia...,NaN,Conference review
4,5,Transformer-Based Models for Natural Language ...,2026,2026 5th International Conference on Electroni...,Transformer architectures have become the domi...,Chinese NLP; large language models; long conte...,Conference paper


In [7]:
!uv pip install ipywidgets jupyterlab --quiet
from tqdm import tqdm
import json
import re

In [8]:
def screening(title, abstract):

    prompt = f"""
        Evaluate the following paper for a Systematic Literature Review.

        Inclusion Criteria:
        {INCLUSION}

        Exclusion Criteria:
        {EXCLUSION}

        Title:
        {title}

        Abstract:
        {abstract}

        Return ONLY a valid JSON object with exactly this format:

        {{
            "decision": "Include | Exclude | Uncertain",
            "confidence": 0,
            "reason": "Short reason (max 30 words)"
        }}
        """

    try:

        response = client.chat.completions.create(
            model="gemma3:4b",
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert systematic literature reviewer. "
                        "Never explain your reasoning. "
                        "Never use markdown. "
                        "Return ONLY valid JSON."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        content = response.choices[0].message.content.strip()

        # Hapus markdown
        content = content.replace("```json", "")
        content = content.replace("```", "")

        # Hapus <think> jika ada
        content = re.sub(
            r"<think>.*?</think>",
            "",
            content,
            flags=re.DOTALL
        ).strip()

        # Ambil JSON pertama
        match = re.search(r"\{.*\}", content, re.DOTALL)

        if match:
            content = match.group(0)

        result = json.loads(content)

        # Validasi field
        result.setdefault("decision", "ERROR")
        result.setdefault("confidence", 0)
        result.setdefault("reason", "")

        return result

    except Exception as e:

        return {
            "decision": "ERROR",
            "confidence": 0,
            "reason": str(e),
            "raw_output": content if "content" in locals() else ""
        }

In [11]:
paper = df.iloc[0]

result = screening(
    paper["Title"],
    paper["Abstract"]
)

result

{'decision': 'Include',
 'confidence': 0.95,
 'reason': 'The study utilizes LLMs in an educational context for biomedical relation extraction, aligning with all inclusion criteria.'}

In [12]:
# results = []

# for _, row in tqdm(df.iterrows(), total=len(df)):

#     result = screening(
#         row["Title"],
#         row["Abstract"]
#     )

# results.append(result)

In [13]:
# result_df = pd.DataFrame(results)

# output = pd.concat(
#     [df, result_df],
#     axis=1
# )

In [14]:
# output

In [15]:
# output.to_csv(
#     "screened_results2.csv",
#     index=False
# )

# print("Finished")

In [18]:
import pandas as pd
import glob
import os
from tqdm import tqdm

# Folder containing the CSV files
folder_path = r"C:\Users\ikaqu\Desktop\PROJECT\automatic-SLR\outputs_wos"

# Get all CSV files
csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

print(f"Found {len(csv_files)} CSV files.")

# Read and combine all CSVs
dfs = []
for file in tqdm(csv_files):
    df = pd.read_csv(file)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# Save the combined CSV
output_file = os.path.join(folder_path, "combined_wos.csv")
combined_df.to_csv(output_file, index=False)

print(f"Combined {len(csv_files)} files into:")
print(output_file)
print(f"Total rows: {len(combined_df)}")

Found 69 CSV files.


100%|██████████| 69/69 [00:00<00:00, 74.81it/s]

Combined 69 files into:
C:\Users\ikaqu\Desktop\PROJECT\automatic-SLR\outputs_wos\combined_wos.csv
Total rows: 344
